# Chapter 7, Guide 2: Simulating a 1-sparse Hamiltonian

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 7,
"Building the sparse simulator" and "Uncomputing and testing the sparse simulator".

The target is the book's 1-sparse example,

$$H=\begin{pmatrix}0&0&0&a\\0&b&0&0\\0&0&c&0\\a&0&0&0\end{pmatrix},\qquad f(0)=3,\ f(1)=1,\ f(2)=2,\ f(3)=0,$$

whose evolution is $e^{-iaXt}\oplus\mathrm{diag}(e^{-ibt},e^{-ict})$ in the basis $(0,3,1,2)$.

Registers, as the guide lists them: two $n$-qubit index registers $x$ and $y=f(x)$, one
comparison ancilla, one equality flag (so the rotation can be $R_z$-type for an eigenstate
and $R_x$ for a pair), a value register holding $H_{xy}$ in fixed point, plus the
borrow scratch of the ripple comparator. Steps: $O_f$, COMP (with $x\le y\mapsto 0$),
controlled-SWAP, $O_H$, the rotation controlled bit-by-bit by the value register, then
uncompute everything but the rotation.

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`; within a
register, index 0 is the most significant bit.

In [1]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

bq = lambda j, tot: tot - 1 - j

n = 2                                  # index register width
m = 3                                  # value register width: v = v0/2 + v1/4 + v2/8
a, b, c = 0.75, 0.5, 0.25              # all exactly representable in 3 fixed-point bits
f = {0: 3, 1: 1, 2: 2, 3: 0}
Hval = {(0, 3): a, (3, 0): a, (1, 1): b, (2, 2): c}

H = np.zeros((4, 4)); H[0, 3] = H[3, 0] = a; H[1, 1] = b; H[2, 2] = c
assert np.allclose(H, H.T) and all(f[f[x]] == x for x in f)
print("H Hermitian, f an involution")


def to_bits(v, width):            # fixed-point fraction -> list of bits, MSB first
    bits = [int(round(v * 2**width)) >> (width - 1 - k) & 1 for k in range(width)]
    assert abs(sum(bit * 2.0**-(k + 1) for k, bit in enumerate(bits)) - v) < 1e-12, "not representable"
    return bits

H Hermitian, f an involution


## Wire layout

In [2]:
layout = (["x0", "x1", "y0", "y1", "bor0", "bor1", "cmp", "eq"] + [f"v{k}" for k in range(m)])
tot = len(layout)
W = {name: bq(i, tot) for i, name in enumerate(layout)}
X = [W["x0"], W["x1"]]; Y = [W["y0"], W["y1"]]; BOR = [W["bor0"], W["bor1"]]
CMP, EQ = W["cmp"], W["eq"]; VAL = [W[f"v{k}"] for k in range(m)]
print(tot, "qubits:", layout)

11 qubits: ['x0', 'x1', 'y0', 'y1', 'bor0', 'bor1', 'cmp', 'eq', 'v0', 'v1', 'v2']


## The pieces

In [3]:
def table_oracle(qc, controls, in_value, targets, out_bits, width):
    """Flip targets[k] where out_bits[k]==1, when the control register holds in_value."""
    bits = [(in_value >> (width - 1 - k)) & 1 for k in range(width)]
    zeros = [controls[k] for k, bit in enumerate(bits) if bit == 0]
    if zeros: qc.x(zeros)
    for tq, ob in zip(targets, out_bits):
        if ob: qc.mcx(controls, tq)
    if zeros: qc.x(zeros)


def O_f(qc):
    """|x>|0> -> |x>|f(x)>, as a lookup table (the in-place CNOT trick of Exercise 7.x
    would also do; the table form is what a general oracle looks like)."""
    for x in range(4):
        table_oracle(qc, X, x, Y, [(f[x] >> 1) & 1, f[x] & 1], n)


def O_H(qc):
    """|x>|y>|0> -> |x>|y>|H_xy> on the value register (fixed point, MSB first)."""
    for (x, y), v in Hval.items():
        idx = (x << n) | y
        table_oracle(qc, X + Y, idx, VAL, to_bits(v, m), 2 * n)


def compare(qc):
    """cmp ^= [x > y], via a ripple-borrow computation of y - x (borrow out = 1 iff y < x).
    eq  ^= [x == y]. Scratch is uncomputed."""
    # t_i = x_i xor y_i, stored in y_i temporarily (LSB is index n-1)
    for i in range(n): qc.cx(X[i], Y[i])
    # equality: all t_i == 0
    qc.x(Y); qc.mcx(Y, EQ); qc.x(Y)
    # borrow chain from the LSB: b_{i+1} = (t_i AND x_i) XOR (NOT t_i AND b_i)
    order = list(range(n - 1, -1, -1))                   # LSB first
    for step, i in enumerate(order):
        bout = BOR[step]
        qc.ccx(Y[i], X[i], bout)
        if step > 0:
            bin_ = BOR[step - 1]
            qc.x(Y[i]); qc.ccx(Y[i], bin_, bout); qc.x(Y[i])
    qc.cx(BOR[n - 1], CMP)
    # uncompute the chain (reverse order) and the xors
    for step, i in reversed(list(enumerate(order))):
        bout = BOR[step]
        if step > 0:
            bin_ = BOR[step - 1]
            qc.x(Y[i]); qc.ccx(Y[i], bin_, bout); qc.x(Y[i])
        qc.ccx(Y[i], X[i], bout)
    for i in range(n): qc.cx(X[i], Y[i])


def cswap_xy(qc):
    for i in range(n): qc.cswap(CMP, X[i], Y[i])


def rotation(qc, t):
    """Off-diagonal (eq=0): R_x(2 H_xy t) on cmp, one controlled rotation per value bit.
    Diagonal (eq=1): the phase e^{-i H_xx t}, one controlled phase per value bit."""
    qc.x(EQ)
    for k, vq in enumerate(VAL):
        qc.mcrx(2 * 2.0**-(k + 1) * t, [EQ, vq], CMP)      # controlled on eq=0 and bit k
    qc.x(EQ)
    for k, vq in enumerate(VAL):
        qc.cp(-2.0**-(k + 1) * t, vq, EQ)                 # phase when eq=1 and bit k=1


def sparse_evolution(t):
    qc = QuantumCircuit(tot)
    O_f(qc); compare(qc); cswap_xy(qc); O_H(qc)
    rotation(qc, t)
    O_H(qc); cswap_xy(qc); compare(qc); O_f(qc)          # uncompute: each piece is self-inverse
    return qc


def x_register_block(t):
    """The unitary on the x register when every other wire starts and ends in |0>."""
    U = Operator(sparse_evolution(t)).data
    N = 2**tot
    # x register = top n book wires = Qiskit qubits tot-1, tot-2 -> most significant bits of the index
    stride = 2**(tot - n)
    idx = np.arange(4) * stride
    return U[np.ix_(idx, idx)]

## Test the pieces: the comparator must send $x\le y$ to 0

In [4]:
def check_compare():
    for x in range(4):
        for y in range(4):
            qc = QuantumCircuit(tot)
            for i in range(n):
                if (x >> (n - 1 - i)) & 1: qc.x(X[i])
                if (y >> (n - 1 - i)) & 1: qc.x(Y[i])
            compare(qc)
            sv = Statevector(qc)
            outcome = list(sv.probabilities_dict().keys())[0]       # Qiskit string, qubit 0 on the right
            s = outcome[::-1]                                         # now index = Qiskit qubit
            cmp_bit, eq_bit = int(s[CMP]), int(s[EQ])
            bor = [int(s[q]) for q in BOR]
            assert cmp_bit == int(x > y) and eq_bit == int(x == y) and bor == [0, 0], (x, y, cmp_bit, eq_bit, bor)
    print("comparator: cmp = [x>y], eq = [x==y], scratch clean, for all 16 inputs")


check_compare()

comparator: cmp = [x>y], eq = [x==y], scratch clean, for all 16 inputs


## Test the whole circuit against `expm(-1j*H*t)`, both cases separately

In [5]:
t = 1.7
U = x_register_block(t)
E = expm(-1j * H * t)
print("max |U - expm|:", np.abs(U - E).max())
assert np.allclose(U, E, atol=1e-10)

# case II: |0> is NOT an eigenstate here; |1> and |2> are. The book's test prepares an
# eigenstate (diagonal, should pick up a phase) and a swapped pair (should rotate).
psi1 = U @ np.eye(4)[1]
print("|1> -> phase e^{-ibt}:", np.allclose(psi1, np.exp(-1j * b * t) * np.eye(4)[1]))
pair = (np.eye(4)[0] + np.eye(4)[3]) / np.sqrt(2)
out = U @ pair
print("(|0>+|3>)/sqrt2 -> rotated by R_x(2at):", np.round(out, 4),
      "  expected", np.round(expm(-1j * a * t * np.array([[0, 1], [1, 0]])) @ np.array([1, 1]) / np.sqrt(2), 4))
print("Test passed: the circuit implements exp(-iHt) = e^{-iaXt} ⊕ diag(e^{-ibt}, e^{-ict})")

max |U - expm|: 3.586553099531655e-14
|1> -> phase e^{-ibt}: True
(|0>+|3>)/sqrt2 -> rotated by R_x(2at): [0.2061-0.6764j 0.    +0.j     0.    +0.j     0.2061-0.6764j]   expected [0.2061-0.6764j 0.2061-0.6764j]
Test passed: the circuit implements exp(-iHt) = e^{-iaXt} ⊕ diag(e^{-ibt}, e^{-ict})


## The two angle traps, reproduced

$R_x(2a)$ instead of $R_x(2at)$ fails; and the rotation must carry $t$ in the angle.

In [6]:
wrong = expm(-1j * a * np.array([[0, 1], [1, 0]]))
right = expm(-1j * a * t * np.array([[0, 1], [1, 0]]))
print("R_x(2a) equals e^{-iaXt}?", np.allclose(wrong, right), " R_x(2at)?", np.allclose(right, U[np.ix_([0, 3], [0, 3])]))

R_x(2a) equals e^{-iaXt}? False  R_x(2at)? True


## Query count

Four oracle queries per 1-sparse piece (two to compute, two to uncompute), and the
circuit simulates any $t$ in one pass since the angle scales with $t$. A $d$-sparse $H$
split into $D$ pieces and Trotterized with $r$ steps costs $4Dr$ queries.